# POSE — `setup.ipynb` · Training von Grund auf reproduzieren

**Ziel:** dieses Notebook von **oben nach unten** durchlaufen, um den
kompletten BOP-Pose-Stack zu reproduzieren — von den CAD-Teilen über die
Isaac-SDG-Daten bis zu den trainierten GDRNPP-Modellen.

Die schweren Stufen (Datengenerierung, Training) laufen auf der **GPU-Box**
(RTX 3090). Dieses Notebook **ruft die Skripte in `box_src/` auf** und
**erklärt** sie — es dupliziert keine Pipeline-Logik. Die eigentliche Arbeit
passiert auf der Box via `box_src/gpu_run.sh` (SSH-Run-Harness mit
Wake-on-LAN). Lange Jobs werden als `nohup` gestartet und gepollt — das
Notebook blockiert nie tagelang.

## Architektur (ADR-018 — Pivot auf BOP-SOTA)

```
CAD-Teile (GLB)  ─gen_models_info.py─▶  models/*.ply (mm) + models_info.json (Symmetrie)
Isaac-Zelle      ─gen_sdg_arm_visible.py─▶  Top-Down-RGB MIT LARA5-Arm + gedroppte Teile
                 ─isaac_to_bop.py / convert_full_to_bop.py─▶  BOP-Datensatz (train/val)
                 ─validate_bop_full.py + vis_bop_overlay.py─▶  bop_toolkit-Gate + GT-Overlay
BOP-Daten        ─train_detector_armvis.py─▶  detector.pt (YOLOv8-OBB, arm-sichtbar)
                 ─obb_to_aabb_dets.py─▶  BOP-Detektionen (OBB→AABB-Bridge)
                 ─train_chain.sh (GDRNPP)─▶  per-Objekt GDRNPP-Checkpoints (RGB-only)
```

**Eingefrorene Konventionen:**
- RGB-only **hart** — keine Depth in den Pose-Netzen (ADR-018).
- Top-Down-View **MIT sichtbarem Arm** — der Arm ist Occluder, nie ein BOP-Objekt.
- `obj_id`-Mapping (1-basiert, eingefroren): `1=Anker_Kurz 2=Anker_Lang
  3=Buerstenhalter_2polig 4=Getriebegehaeuse_typ4 5=Ringmagnet 6=Zahnrad`.
- Symmetrie: Anker/Ring = continuous um Y, Zahnrad = discrete C_7.
- Detektor-Klasse (0-basiert) `+1` = `obj_id` (1-basiert).

## Voraussetzungen

| Was | Wert |
|---|---|
| GPU-Box | `max@100.85.216.95` (Tailscale), RTX 3090 24 GB |
| Box-Repo | `/mnt/data/kip_pose` (= `BOX_REPO`) |
| Isaac-venv | `/mnt/data/isaacsim-venv` (SDG-Gen) |
| Train-venv | `/mnt/data/train-venv` (YOLOv8-OBB-Detektor) |
| bop-venv | `/mnt/data/bop/bop-venv` (Konverter, Validierung, Eval) |
| gdrnpp-venv | `/mnt/data/bop/gdrnpp-venv` (GDRNPP, isolierte Pins) |
| Lokal | nur `bash`/`ssh` für den Run-Harness; alles Schwere läuft remote |

> Detaillierte Box-Einrichtung: `box_src/BOP_SETUP.md`. Eval: `box_src/EVAL_BOP.md`.

**Hinweis:** wenn auf der Box gerade ein Training läuft (`train_chain.sh` als
`nohup`), stören die Zellen hier es **nicht** — sie lesen nur Logs bzw. starten
Stufen, die du bewusst auslöst. Eine RTX 3090 = Stufen laufen sequenziell.

## 1 · Config + Box-Verbindung

Alle Pfade kommen aus `project/.env`. Der Run-Harness `box_src/gpu_run.sh`
weckt die Box bei Bedarf (Wake-on-LAN über den Raspberry-Pi-Relay), wartet
auf SSH und führt **einen** Befehl auf der Box aus. Lange Jobs wrappst du im
Remote-Command selbst in `nohup ... &` (der Harness daemonisiert nicht).

In [ ]:
import os, subprocess, pathlib, time, textwrap

# Worktree-Root + zentrale Pfade (CWD-unabhängig).
PROJECT = pathlib.Path.cwd()
if PROJECT.name != 'project':
    PROJECT = next((p for p in [PROJECT/'project', *PROJECT.parents] if (p/'e2e_infer.py').exists()), PROJECT)
ROOT     = PROJECT.parent              # Worktree-Root (enthält box_src/)
BOX_SRC  = ROOT / 'box_src'
GPU_RUN  = BOX_SRC / 'gpu_run.sh'
ENV_FILE = PROJECT / '.env'

# .env einlesen (KEY=VALUE Zeilen).
ENV = {}
for line in ENV_FILE.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        k, v = line.split('=', 1); ENV[k.strip()] = v.strip()

BOX_REPO     = ENV.get('BOX_REPO', '/mnt/data/kip_pose')
BOX_ISAAC_PY = ENV.get('BOX_ISAAC_PY', '/mnt/data/isaacsim-venv/bin/python')
BOX_TRAIN_PY = ENV.get('BOX_TRAIN_PY', '/mnt/data/train-venv/bin/python')
BOP_VENV_PY  = '/mnt/data/bop/bop-venv/bin/python'
GDRN_VENV_PY = '/mnt/data/bop/gdrnpp-venv/bin/python'
BOP_ROOT     = f'{BOX_REPO}/project/bop/pose_isaac'   # BOP-Datensatz auf der Box
LOGDIR       = '/mnt/data/bop/logs'

print('PROJECT  :', PROJECT)
print('BOX_SRC  :', BOX_SRC)
print('BOX_REPO :', BOX_REPO)
print('BOP_ROOT :', BOP_ROOT)

In [ ]:
def box(cmd, workdir=None, pull=None, timeout=None, check=False):
    """Einen Befehl auf der GPU-Box ausführen (via box_src/gpu_run.sh).

    cmd      : Remote-Befehl (String). Für lange Jobs INNEN 'nohup ... &' setzen.
    workdir  : optionales -d Arbeitsverzeichnis auf der Box (Default: BOX_REPO).
    pull     : optionales 'REMOTE:LOCAL' rsync-Pull nach dem Lauf.
    check    : True -> RuntimeError bei rc!=0.
    Gibt (rc, stdout) zurück und druckt die Ausgabe."""
    argv = ['bash', str(GPU_RUN)]
    if workdir: argv += ['-d', workdir]
    if pull:    argv += ['-p', pull]
    if timeout: argv += ['-t', str(timeout)]
    argv += ['--', cmd]
    p = subprocess.run(argv, capture_output=True, text=True)
    out = (p.stdout or '') + (p.stderr or '')
    print(out.rstrip())
    if check and p.returncode != 0:
        raise RuntimeError(f'box command rc={p.returncode}: {cmd}')
    return p.returncode, p.stdout

def poll(logpath, proc_pattern, lines=40):
    """Tail eines nohup-Logs + Prozess-Check. Wiederhol-aufrufbar (re-run die Zelle)."""
    box(f"tail -n {lines} {logpath}; pgrep -f {proc_pattern!r} >/dev/null && echo '>>> RUNNING' || echo '>>> EXITED'")

print('Helper bereit: box(cmd, ...) und poll(logpath, pattern).')

### Verbindungs-Smoke

Weckt die Box (falls nötig) und prüft die GPU. Wenn hier `NVIDIA GeForce RTX
3090` erscheint, ist die Verbindung gut. Auf einer geteilten GPU zeigt
`nvidia-smi` auch ein evtl. laufendes Training — **nicht stören**.

In [ ]:
box('nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader')
# Falls gerade ein Training läuft, sieht man hier die Belegung. Nur lesen.

## 2 · Isaac — arm-sichtbare SDG-Daten generieren

`box_src/gen_sdg_arm_visible.py` rendert die echte GST-Zelle aus einer
Top-Down-Zivid-artigen Kamera — **mit sichtbarem LARA5-Arm + Wagen** als
Occluder. Teile (Fokus: Anker_Kurz, Anker_Lang, Zahnrad; Rest als
Distraktoren) werden mit Physik gedroppt, setzen sich auf der kalibrierten
Wagenoberfläche, und werden zusammen mit allem aufgenommen, was der
Isaac→BOP-Konverter braucht (RGB, Depth (nur für GT-Maskenbau), Instanz-/
Semantik-Seg + Labels, `gt_raw` mit Intrinsics + cam_c2w + T_obj2world,
`obb_2d`/`bbox_2d` für den Detektor).

Läuft im **Isaac-venv** auf der Box. Volle 2000-Frame-Generierung dauert; als
`nohup` starten und pollen.

In [ ]:
# Parameter der SDG-Generierung (anpassen nach Bedarf).
SDG_OUT    = f'{BOX_REPO}/data/sdg_armvis_full'    # Ziel-Ordner auf der Box
SDG_FRAMES = 2000                                   # Frame-Anzahl (voller Lauf)
SDG_LOG    = f'{LOGDIR}/sdg_armvis.log'

sdg_cmd = textwrap.dedent(f'''
    mkdir -p {LOGDIR} {SDG_OUT}
    nohup {BOX_ISAAC_PY} {BOX_REPO}/box_src/gen_sdg_arm_visible.py \\
        --output {SDG_OUT} --num-frames {SDG_FRAMES} \\
        > {SDG_LOG} 2>&1 &
    echo "SDG_PID=$!"
''').strip()
print(sdg_cmd)
# Ausführen: die nächste Zelle. (gen_sdg_arm_visible.py --help zeigt alle Flags.)

In [ ]:
# >>> SDG-Generierung starten (nohup, non-blocking). <<<
# box(sdg_cmd)
print('Zum Starten die Zeile oben einkommentieren. Auskommentiert lassen, wenn die Daten schon existieren.')

In [ ]:
# Fortschritt pollen (Zelle beliebig oft re-runnen).
# poll(SDG_LOG, 'gen_sdg_arm_visible')
# Wenn '>>> EXITED' + die letzten Log-Zeilen 'DONE' o.ä. zeigen: Daten fertig.
print('Poll-Zelle — einkommentieren wenn ein SDG-Job läuft.')

## 3 · Isaac → BOP konvertieren, train/val-Split, validieren, GT-Overlay

Der zentrale Daten-Contract (Viktor §1). `convert_full_to_bop.py` mischt die
2000 Roh-Frames deterministisch ~90/10 in `train_pbr`/`val`, gruppiert sie in
BOP-Szenen und ruft pro Szene `isaac_to_bop.py` auf — Ergebnis ist ein
BOP-Datensatz, den **CNOS / GigaPose / MegaPose / GDRNPP / bop_toolkit ohne
eine Zeile Repo-Patch** lesen (`scene_camera/gt/gt_info.json`, `mask_visib/`,
`models/*.ply` in mm). Läuft in der **bop-venv** (numpy + PIL, kein Isaac).

In [ ]:
# 3a · Konvertierung Isaac-Bundle -> BOP-Datensatz mit train/val-Split.
conv_cmd = textwrap.dedent(f'''
    nohup {BOP_VENV_PY} {BOX_REPO}/box_src/convert_full_to_bop.py \\
        --src {SDG_OUT} --bop-root {BOP_ROOT} \\
        > {LOGDIR}/convert_bop.log 2>&1 &
    echo "CONV_PID=$!"
''').strip()
print(conv_cmd)
# box(conv_cmd)        # starten
# poll(f'{LOGDIR}/convert_bop.log', 'convert_full_to_bop')   # pollen

### 3b · Validierung mit bop_toolkit (Native-Load-Gate)

`validate_bop_full.py` lädt den Datensatz **mit genau den bop_toolkit-Loadern**,
die alle Downstream-Repos benutzen, prüft Szenen-/Frame-/GT-Counts, löst die
Symmetrien über `models_info.json` auf, belegt die Arm-Occlusion über die
`visib_fract`-Verteilung und prüft die Masken. Exit 0 + `VALIDATE_BOP_OK` nur
wenn alles passt.

In [ ]:
validate_cmd = (
    f'PYTHONPATH=/mnt/data/bop/repos/bop_toolkit {BOP_VENV_PY} '
    f'{BOX_REPO}/box_src/validate_bop_full.py --bop-root {BOP_ROOT}'
)
print(validate_cmd)
# box(validate_cmd, check=True)   # blockiert kurz; VALIDATE_BOP_OK = Daten gültig

### 3c · GT-Pose-Overlay (visueller Abnahme-Check)

`vis_bop_overlay.py` projiziert jedes PLY-Mesh an seiner `scene_gt`-Pose auf
das RGB. Stimmt die Konverter-Mathematik (mm, row-major, w2c-Inversion,
GL→CV-Flip), landen die Punkte exakt auf den Teilen — der Arm verdeckt Teile
sichtbar. Ein paar Overlays holen wir lokal herunter und zeigen sie inline.

In [ ]:
OVERLAY_REMOTE = f'{LOGDIR}/gt_overlay'
OVERLAY_LOCAL  = PROJECT / 'temp' / 'gt_overlay'
overlay_cmd = (
    f'{BOP_VENV_PY} {BOX_REPO}/box_src/vis_bop_overlay.py '
    f'--bop-root {BOP_ROOT} --split val --num 4 --out {OVERLAY_REMOTE}'
)
print(overlay_cmd)
# box(overlay_cmd, pull=f'{OVERLAY_REMOTE}/:{OVERLAY_LOCAL}/')   # erzeugt + pullt

In [ ]:
# Heruntergeladene Overlays inline anzeigen (sofern vorhanden).
from IPython.display import Image as IPyImage, display
pngs = sorted(OVERLAY_LOCAL.glob('overlay_*.png')) if OVERLAY_LOCAL.exists() else []
if pngs:
    for p in pngs[:4]:
        print(p.name); display(IPyImage(filename=str(p)))
else:
    print('Noch keine Overlays lokal. Erst 3c oben ausführen (box+pull).')

## 4 · `models_info.json` mit Symmetrie-Flags

`gen_models_info.py` exportiert pro Teil das GLB-Mesh nach `models/obj_*.ply`
in **mm** (GLB ist Meter → ×1000), berechnet `diameter/min_*/size_*` und
schreibt die **Symmetrie-Flags** (Viktor §2 — die analytische Lösung des
120°/91°-Problems):

- `Anker_Kurz`, `Anker_Lang`, `Ringmagnet` → `symmetries_continuous`, Achse Y `[0,1,0]`
- `Zahnrad` → `symmetries_discrete`, **C_N** (N = Zähnezahl, aus dem Mesh gezählt)
- `Buerstenhalter_2polig`, `Getriebegehaeuse_typ4` → keine

Diese `models_info.json` füttert sowohl GDRNPP als auch die symmetrie-bewusste
BOP-Eval. Läuft in der bop-venv (trimesh + numpy).

In [ ]:
GLB_DIR = f'{BOX_REPO}/project/frontend/assets/parts'   # echte CAD-GLBs
models_cmd = (
    f'{BOP_VENV_PY} {BOX_REPO}/box_src/gen_models_info.py '
    f'--glb-dir {GLB_DIR} --out {BOP_ROOT}/models'
)
print(models_cmd)
# box(models_cmd, check=True)
# Danach: cat models_info.json -> Symmetrie-Flags prüfen.
# box(f'{BOP_VENV_PY} -c "import json;d=json.load(open(\'{BOP_ROOT}/models/models_info.json\'));print(json.dumps(d,indent=2)[:800])"')

## 5 · Detektor-Retrain MIT Arm

Der alte `detector.pt` war **arm-versteckt** trainiert (mAP50 0.987 auf der
leichteren arm-freien Verteilung) und bricht unter Arm-Occlusion.
`train_detector_armvis.py` trainiert YOLOv8-OBB auf der **arm-sichtbaren**
SDG-Verteilung neu. Die Klassen-Reihenfolge ist **eingefroren** auf das
globale `obj_id`-Mapping (`anker_kurz=0 … zahnrad=5`), damit
`category_id + 1 == obj_id` immer gilt (§1.2/§4.1).

`obb_to_aabb_dets.py` ist danach die **OBB→AABB-Bridge**: es fährt den
trainierten OBB-Detektor über den `val`-Split und schreibt die BOP-Default-
Detektionen (achsenparallele Box + `category_id=obj_id`), die GDRNPP als Input
konsumiert.

Läuft in der **train-venv**. ~1 h. Auf einer GPU **nicht** parallel zu GDRNPP
starten — `train_chain.sh` (Stufe 6) kettet das ohnehin korrekt.

In [ ]:
DETOUT = f'{BOX_REPO}/data/detector_armvis'
det_cmd = textwrap.dedent(f'''
    nohup {BOX_TRAIN_PY} {BOX_REPO}/box_src/train_detector_armvis.py \\
        --src {SDG_OUT} --out {DETOUT} \\
        --epochs 100 --imgsz 1280 --batch 8 --max-occ 0.85 \\
        > {LOGDIR}/detector_armvis.log 2>&1 &
    echo "DET_PID=$!"
''').strip()
print(det_cmd)
# box(det_cmd); poll(f'{LOGDIR}/detector_armvis.log', 'train_detector_armvis')

# OBB->AABB-Bridge auf den val-Split (nach dem Detektor-Training):
bridge_cmd = (
    f'{BOX_TRAIN_PY} {BOX_REPO}/box_src/obb_to_aabb_dets.py '
    f'--weights {DETOUT}/detector.pt --bop-root {BOP_ROOT} --split val '
    f'--out {BOP_ROOT}/val/det_obb2aabb_pose_isaac_val.json --conf 0.1 --imgsz 1280'
)
print('\n' + bridge_cmd)
# box(bridge_cmd)

## 6 · GDRNPP RGB-Training (Gleis B, max Genauigkeit)

GDRNPP wird **per Objekt** auf den Isaac-PBR-Synth-Daten trainiert — RGB-only.
Da eine RTX 3090 verfügbar ist, müssen Detektor-Retrain und GDRNPP-Training
**sequenziell** laufen. `train_chain.sh` macht genau das: Detektor (~1 h) →
OBB→AABB-Bridge → GDRNPP-Deploy → GDRNPP-SO-Training für `anker_kurz`,
`anker_lang`, `zahnrad` (je ~1–2 Tage).

**So startet man die volle Kette** (genau so läuft sie aktuell auf der Box):

```bash
cd /mnt/data/kip_pose
nohup bash box_src/train_chain.sh > /mnt/data/bop/logs/train_chain.log 2>&1 &
echo "CHAIN_PID=$!"
```

> ⚠️ Wenn die Kette **schon läuft**, NICHT neu starten — nur pollen (unten).

In [ ]:
CHAIN_LOG = f'{LOGDIR}/train_chain.log'
chain_cmd = textwrap.dedent(f'''
    cd {BOX_REPO}
    nohup bash box_src/train_chain.sh > {CHAIN_LOG} 2>&1 &
    echo "CHAIN_PID=$!"
''').strip()
print(chain_cmd)
# >>> NUR ausführen, wenn KEINE Kette läuft. <<<
# box(chain_cmd)

In [ ]:
# Trainings-Fortschritt pollen (Zelle beliebig oft re-runnen — non-destruktiv).
poll(CHAIN_LOG, 'train_chain', lines=40)
# GPU-Auslastung dazu:
box('nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader')

### Wann ist GDRNPP fertig?

Im `train_chain.log` zeigt `TRAIN_CHAIN_DONE (detector + 3 GDRNPP SO models)`
den Abschluss. Die Checkpoints liegen unter
`/mnt/data/bop/repos/gdrnpp/output/.../<obj>/...` (`.pth`). Diese Pfade
trägst du dann in **`infer.ipynb`** als `GDRNPP_CHECKPOINT` ein — bis dahin
läuft die Inferenz im klar markierten **MOCK-Modus** und der 3D-Viewer zeigt
trotzdem plausible Posen.

## Optimierungen & Empfehlungen

**Angewandt (sicher, kein laufendes Training berührt):**
- Teile-Mapping zentralisiert: `e2e_infer.available_parts()` zieht jetzt aus
  `bop_adapter.OBJ_ID_TO_PART` (Single-Source) statt einer driftenden
  Hardcoded-Liste / Datei-Abhängigkeit.
- `box()`/`poll()`-Helfer kapseln den Run-Harness sauber, mit
  Fehler-Weitergabe (`check=True`) und re-runnbarem Polling.

**Empfohlen (nächste Iterationen):**
- **Mehr Daten / stärkere DR:** GDRNPP-Synth-only skaliert mit Daten +
  Domain-Randomization — mehr Frames, mehr Material-/Licht-/Pose-Varianz
  (wirksamster Sim2Real-Hebel auf texturlosem Metall).
- **Epochen/Schedule:** GDRNPP-SO-Configs (`configs/gdrn/poseIsaacPbrSO/*.py`)
  längere Schedules geben bei texturlosen Teilen meist noch AR.
- **RGB-vs-RGB-D-Ablation** mit confidence-gefilterter Zivid-Depth
  (BOP-Industrial-Evidenz: +10–15 AR) — als optionales Experiment, RGB bleibt
  der harte Default.
- **Eval-Automatisierung:** nach jedem GDRNPP-Checkpoint automatisch
  `box_src/eval_bop.sh --preds <csv>` triggern + Report archivieren.
- **Zahnrad-N verifizieren:** `gen_models_info.py` zählt N aus dem Mesh —
  einmal gegen das CAD gegenchecken (Symmetrie C_7 angenommen).